# Text Normalization

#### This tutorial covers the following 
##### - reading file from a url
##### - stripping file to keep the relevant content 
##### - converting text to lowercase
##### - expansion of contraction 
##### - expansion of slangs
##### - simple tokenization
##### - stop words filtering
##### - part of speech tagging
##### - lemmatization
##### - stemming (porter and snowball)
##### - subword tokenization (BPE and WordPiece)

In [1]:
import requests
import re
import string
import nltk
from nltk.corpus import stopwords
# download stopwords if not already present
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/shailendra/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
# Download Frankenstein from Project Gutenberg
url = "https://www.gutenberg.org/cache/epub/42324/pg42324.txt"
response = requests.get(url)
response.raise_for_status() ## checks the HTTP response status code
raw_text = response.text

In [3]:
# sample text
raw_text[:500]

'\ufeffThe Project Gutenberg eBook of Frankenstein; Or, The Modern Prometheus\r\n    \r\nThis ebook is for the use of anyone anywhere in the United States and\r\nmost other parts of the world at no cost and with almost no restrictions\r\nwhatsoever. You may copy it, give it away or re-use it under the terms\r\nof the Project Gutenberg License included with this ebook or online\r\nat www.gutenberg.org. If you are not located in the United States,\r\nyou will have to check the laws of the country where you are locate'

In [4]:
# remove gutenberg boilerplate
start_marker = "CHAPTER I."
end_marker   = "THE END."

start = raw_text.find(start_marker)
end   = raw_text.find(end_marker)

if start != -1 and end != -1:
    text = raw_text[start+len(start_marker):end].strip()   #.strip() removes leading and trailing whitespace from a string
else:
    text = raw_text

In [5]:
text[:500]

'I am by birth a Genevese; and my family is one of the most distinguished\r\nof that republic. My ancestors had been for many years counsellors and\r\nsyndics; and my father had filled several public situations with honour\r\nand reputation. He was respected by all who knew him, for his integrity\r\nand indefatigable attention to public business. He passed his younger\r\ndays perpetually occupied by the affairs of his country; a variety of\r\ncircumstances had prevented his marrying early, nor was it until t'

In [6]:
# Text Normalization
## lowercase
## contraction expansion
## slang abberrivation handling
#  remove punctuation, numbers, special characters
#  remove stop words

# lowercase
text_lower = text.lower()

In [7]:
# contraction expansion

contractions = {
    "i'm": "i am", "you're": "you are", "he's": "he is", "she's": "she is",
    "it's": "it is", "we're": "we are", "they're": "they are",
    "isn't": "is not", "aren't": "are not", "wasn't": "was not",
    "weren't": "were not", "don't": "do not", "doesn't": "does not",
    "didn't": "did not", "can't": "cannot", "won't": "will not",
    "couldn't": "could not", "shouldn't": "should not", "wouldn't": "would not",
    "there's": "there is", "that's": "that is", "what's": "what is",
    "who's": "who is", "let's": "let us"
}

def expand_contractions(text):
    for c, full in contractions.items():
        text = re.sub(rf"\b{c}\b", full, text, flags=re.IGNORECASE)
    return text

text = expand_contractions(text.lower())

In [8]:
# slang / abbreviation handling
slang_map = {
    "tho": "though",
    "thru": "through",
    "u": "you",
    "ur": "your",
    "btw": "by the way",
    "idk": "i do not know",
    "lol": "laughing out loud"
}

def replace_slang(text):
    for slang, full in slang_map.items():
        text = re.sub(rf"\b{slang}\b", full, text, flags=re.IGNORECASE)
    return text

text = replace_slang(text)

In [9]:
# remove punctuation, numbers, special characters
text = re.sub(r"[^a-z\s]", " ", text)
text = re.sub(r"\s+", " ", text).strip()

In [10]:
text[:500]

'i am by birth a genevese and my family is one of the most distinguished of that republic my ancestors had been for many years counsellors and syndics and my father had filled several public situations with honour and reputation he was respected by all who knew him for his integrity and indefatigable attention to public business he passed his younger days perpetually occupied by the affairs of his country a variety of circumstances had prevented his marrying early nor was it until the decline of '

In [11]:
# simple tokenization
tokens = text.split()

In [12]:
tokens[:20]

['i',
 'am',
 'by',
 'birth',
 'a',
 'genevese',
 'and',
 'my',
 'family',
 'is',
 'one',
 'of',
 'the',
 'most',
 'distinguished',
 'of',
 'that',
 'republic',
 'my',
 'ancestors']

In [13]:
# remove stop words (optiional - be careful)
stop_words = set(stopwords.words("english"))
filtered_tokens = [word for word in tokens if word not in stop_words]

In [14]:
filtered_tokens[:10]

['birth',
 'genevese',
 'family',
 'one',
 'distinguished',
 'republic',
 'ancestors',
 'many',
 'years',
 'counsellors']

In [15]:
# POS (part of speech) tagging 
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.corpus import wordnet
from nltk import pos_tag         # Penn Treebank Tagger
nltk.download("averaged_perceptron_tagger_eng")


def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN

tagged = pos_tag(filtered_tokens)

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/shailendra/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/shailendra/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/shailendra/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [16]:
tagged[:5]

[('birth', 'NN'),
 ('genevese', 'JJ'),
 ('family', 'NN'),
 ('one', 'CD'),
 ('distinguished', 'VBN')]

In [17]:
# lemmatization
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

lemmatized_tokens = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in tagged]

In [18]:
lemmatized_tokens[:5]

['birth', 'genevese', 'family', 'one', 'distinguish']

In [19]:
# stemming

from nltk.stem import PorterStemmer
from nltk.stem import SnowballStemmer

stemmer_p = PorterStemmer()
stemmer_s = SnowballStemmer(language='english')

stemmed_tokens_p = [stemmer_p.stem(word) for word in filtered_tokens]
stemmed_tokens_s = [stemmer_s.stem(word) for word in filtered_tokens]

In [20]:
stemmed_tokens_p[:10]

['birth',
 'geneves',
 'famili',
 'one',
 'distinguish',
 'republ',
 'ancestor',
 'mani',
 'year',
 'counsellor']

In [21]:
stemmed_tokens_s[:10]

['birth',
 'geneves',
 'famili',
 'one',
 'distinguish',
 'republ',
 'ancestor',
 'mani',
 'year',
 'counsellor']

In [22]:
# Join back into text
stemmed_text = " ".join(stemmed_tokens_s)
lemmatized_text = " ".join(lemmatized_tokens)

# Save outputs
with open("frankenstein_stemmed.txt", "w", encoding="utf-8") as f:
    f.write(stemmed_text)

with open("frankenstein_lemmatized.txt", "w", encoding="utf-8") as f:
    f.write(lemmatized_text)

In [23]:
# Show sample
print("=== Sample (Original Normalized) ===")
print(text[:300])

print("\n=== Sample (Stemmed) ===")
print(stemmed_text[:300])

print("\n=== Sample (Lemmatized) ===")
print(lemmatized_text[:300])

=== Sample (Original Normalized) ===
i am by birth a genevese and my family is one of the most distinguished of that republic my ancestors had been for many years counsellors and syndics and my father had filled several public situations with honour and reputation he was respected by all who knew him for his integrity and indefatigable

=== Sample (Stemmed) ===
birth geneves famili one distinguish republ ancestor mani year counsellor syndic father fill sever public situat honour reput respect knew integr indefatig attent public busi pass younger day perpetu occupi affair countri varieti circumst prevent marri earli declin life becam husband father famili c

=== Sample (Lemmatized) ===
birth genevese family one distinguish republic ancestor many year counsellor syndics father fill several public situation honour reputation respect knew integrity indefatigable attention public business pass young day perpetually occupied affair country variety circumstance prevent marry early decli


## Subword Tokenization

#### Byte-Pair Encoding (BPE)

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

In [37]:
len(set(text.split()))

6671

In [38]:
# tokenzier training requires a text file. we are writing text as frankenstein.txt. 
with open("frankenstein.txt", "w", encoding="utf-8") as f:f.write(text)

# Initialize tokenizer
bpe_tokenizer = Tokenizer(BPE(unk_token="[UNK]")) ## unknown tokens are marked as "UNK"
bpe_tokenizer.pre_tokenizer = Whitespace()  ## perform basic tokenziation by spitting words.


# Trainer
bpe_trainer = BpeTrainer(
    vocab_size=3000,
    min_frequency=2,
    special_tokens=["[UNK]"]
)

# Train
bpe_tokenizer.train(["frankenstein.txt"], trainer=bpe_trainer)

In [39]:
text_sample = text[:100]

output = bpe_tokenizer.encode(text_sample)
print(output.tokens)

['i', 'am', 'by', 'birth', 'a', 'genev', 'es', 'e', 'and', 'my', 'family', 'is', 'one', 'of', 'the', 'most', 'distinguished', 'of', 'that', 'rep', 'ublic', 'my', 'ances', 'tor']


#### WordPiece

In [42]:
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer

wp_tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
wp_tokenizer.pre_tokenizer = Whitespace()

wp_trainer = WordPieceTrainer(
    vocab_size=2000,
    min_frequency=2,
    special_tokens=["[UNK]"]
)

wp_tokenizer.train(["frankenstein.txt"], trainer=wp_trainer)

In [43]:
output = wp_tokenizer.encode(text_sample)
print(output.tokens)

['i', 'am', 'by', 'bir', '##th', 'a', 'genev', '##es', '##e', 'and', 'my', 'family', 'is', 'one', 'of', 'the', 'most', 'dist', '##inguished', 'of', 'that', 'rep', '##ub', '##lic', 'my', 'an', '##ces', '##t', '##or']
